In [23]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('sparkp2').getOrCreate()

In [24]:
df = spark.read.csv('delivery_delay_dataset.csv', header=True, inferSchema=True)
df.show(5)

+-----------+----+-------------+------+
|order_count|rain|delivery_time| delay|
+-----------+----+-------------+------+
|         66|   0|         58.4|  Late|
|         50|   1|         45.0|  Late|
|         37|   0|         36.8|OnTime|
|         79|   1|         72.6|  Late|
|         26|   0|         27.4|OnTime|
+-----------+----+-------------+------+
only showing top 5 rows



In [25]:
df.printSchema()

root
 |-- order_count: integer (nullable = true)
 |-- rain: integer (nullable = true)
 |-- delivery_time: double (nullable = true)
 |-- delay: string (nullable = true)



In [26]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

label_indexer = StringIndexer(
    inputCol='delay',
    outputCol='label'
)

feature_cols = [
    'order_count',
    "rain",
    "delivery_time"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=20
)

pipeline = Pipeline(stages=[
    label_indexer,
    assembler,
    rf
])

train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

model = pipeline.fit(train_df)

predictions = model.transform(test_df)

predictions.show(3)

+-----------+----+-------------+------+-----+--------------+--------------------+--------------------+----------+
|order_count|rain|delivery_time| delay|label|      features|       rawPrediction|         probability|prediction|
+-----------+----+-------------+------+-----+--------------+--------------------+--------------------+----------+
|          5|   0|         13.0|OnTime|  1.0|[5.0,0.0,13.0]|[0.04771922097537...|[0.00238596104876...|       1.0|
|          5|   0|         21.0|OnTime|  1.0|[5.0,0.0,21.0]|[0.04771922097537...|[0.00238596104876...|       1.0|
|          5|   0|         26.0|OnTime|  1.0|[5.0,0.0,26.0]|[0.04771922097537...|[0.00238596104876...|       1.0|
+-----------+----+-------------+------+-----+--------------+--------------------+--------------------+----------+
only showing top 3 rows



In [8]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)
print(accuracy)

0.9963570127504554


In [10]:
from pyspark.sql.functions import split, col, avg,count

raw_stream = spark.readStream \
    .format("socket") \
    .option("host", 'localhost') \
    .option("port", 9999) \
    .load()

split_cols = split(raw_stream.value, ",")

stream_df = raw_stream.select(
    split_cols.getItem(0).cast("integer").alias('order_count'),
    split_cols.getItem(1).cast("integer").alias('rain'),
    split_cols.getItem(2).cast("double").alias('delivery_time')
)

predictions = model.transform(stream_df)

output = predictions.select(
    "order_count",
    "rain",
    "delivery_time",
    "prediction"
)


query = output.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

query.awaitTermination()

26/05/15 08:38:46 WARN TextSocketSourceProvider: The socket source should not be used for production applications! It does not support recovery.
26/05/15 08:38:46 WARN StringIndexerModel: Input column delay does not exist during transformation. Skip StringIndexerModel for this column.
26/05/15 08:38:46 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-f12ec28e-a479-4956-9bcb-613095fb8b15. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/15 08:38:46 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+----+-------------+----------+
|order_count|rain|delivery_time|prediction|
+-----------+----+-------------+----------+
+-----------+----+-------------+----------+

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+----+-------------+----------+
|order_count|rain|delivery_time|prediction|
+-----------+----+-------------+----------+
|22         |1   |45.0         |0.0       |
+-----------+----+-------------+----------+

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+----+-------------+----------+
|order_count|rain|delivery_time|prediction|
+-----------+----+-------------+----------+
|55         |1   |67.0         |0.0       |
+-----------+----+-------------+----------+



ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/home/asish-jose/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/asish-jose/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [11]:
df.select("order_count", "delivery_time").show(3)

+-----------+-------------+
|order_count|delivery_time|
+-----------+-------------+
|         66|         58.4|
|         50|         45.0|
|         37|         36.8|
+-----------+-------------+
only showing top 3 rows



In [27]:
df.show(5)

+-----------+----+-------------+------+
|order_count|rain|delivery_time| delay|
+-----------+----+-------------+------+
|         66|   0|         58.4|  Late|
|         50|   1|         45.0|  Late|
|         37|   0|         36.8|OnTime|
|         79|   1|         72.6|  Late|
|         26|   0|         27.4|OnTime|
+-----------+----+-------------+------+
only showing top 5 rows



In [28]:
df.filter(df['rain']==1).show(3)

+-----------+----+-------------+-----+
|order_count|rain|delivery_time|delay|
+-----------+----+-------------+-----+
|         50|   1|         45.0| Late|
|         79|   1|         72.6| Late|
|         28|   1|         51.2| Late|
+-----------+----+-------------+-----+
only showing top 3 rows



In [29]:
df.orderBy("delivery_time").show(3)

+-----------+----+-------------+------+
|order_count|rain|delivery_time| delay|
+-----------+----+-------------+------+
|          5|   0|          9.0|OnTime|
|          5|   0|         10.0|OnTime|
|          8|   0|         10.2|OnTime|
+-----------+----+-------------+------+
only showing top 3 rows



In [30]:
df.orderBy(df.delivery_time.desc()).show(3)

+-----------+----+-------------+-----+
|order_count|rain|delivery_time|delay|
+-----------+----+-------------+-----+
|         98|   1|         89.2| Late|
|        100|   1|         87.0| Late|
|         98|   1|         84.2| Late|
+-----------+----+-------------+-----+
only showing top 3 rows



In [31]:
df = df.withColumn("timeperorder", df.delivery_time/df.order_count).show(3)

+-----------+----+-------------+------+------------------+
|order_count|rain|delivery_time| delay|      timeperorder|
+-----------+----+-------------+------+------------------+
|         66|   0|         58.4|  Late|0.8848484848484848|
|         50|   1|         45.0|  Late|               0.9|
|         37|   0|         36.8|OnTime|0.9945945945945945|
+-----------+----+-------------+------+------------------+
only showing top 3 rows



In [20]:
from pyspark.sql.functions import avg, max, min

df.groupBy("category").agg(
    avg("price").alias("avg_price"),
    max("price").alias("max_price"),
    min("price").alias("min_price")
).show()

AttributeError: 'NoneType' object has no attribute 'select'